<a href="https://colab.research.google.com/github/saacheeludhani/citeforge/blob/main/CiteForge_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install google-auth==2.49.0

In [21]:
from getpass import getpass
from google import genai

api_key = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=api_key)

print("Gemini connected!")

Paste your Gemini API key: ··········
Gemini connected!


In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Give me 3 questions about AI in healthcare."
)

print(response.text)

Here are 3 thought-provoking questions about AI in healthcare, covering clinical application, ethics, and equity:

1. **Liability & Ethics:** If an AI algorithm makes a incorrect diagnostic recommendation that leads to harm, who should be held legally and ethically accountable—the software developer, the healthcare system, or the attending physician who acted on the AI's advice?

2. **Bias & Accuracy:** How can medical researchers ensure that AI training models don't perpetuate historical racial, gender, or socioeconomic biases present in existing healthcare data?

3. **Global Health Equity:** Will the integration of advanced AI in medicine widen the quality-of-care gap between wealthy and low-resource nations, or can it be used as a tool to democratize access to high-quality diagnostics worldwide?


In [ ]:
import json
from google.genai.errors import APIError

def generate_citeforge_content(topic):
    prompt = f"""
You are the content-planning assistant for CiteForge.

Topic: {topic}

Generate 10 varied questions people might ask about this topic.
Include factual, explanatory, practical, benefits, and risks questions.

For each question, write one concise draft claim answering it.
Each claim must express one main idea.
Do not invent source URLs or claim that anything is verified.

Return only valid JSON in this structure:
{{
  "topic": "topic name",
  "questions": [
    {{
      "id": 1,
      "question": "A question about the topic",
      "claim": "A concise draft claim",
      "verification_status": "unverified_draft"
    }}
  ]
}}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    result = json.loads(response.text)
    return result


topic = input("Enter a topic for CiteForge: ").strip()

if topic:
    try:
        result = generate_citeforge_content(topic)

        print("TOPIC:", result["topic"])
        print("QUESTIONS GENERATED:", len(result["questions"]))

        for item in result["questions"]:
            print(f"\n{item['id']}. {item['question']}")
            print("Draft claim:", item["claim"])
            print("Status:", item["verification_status"])
    except APIError as e:
        print(f"\nAPI Error: {e.message}")
        print("The Gemini service is temporarily unavailable or experiencing high demand. Please wait a moment and try running this cell again.")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
else:
    print("Please enter a topic.")

Enter a topic for CiteForge: AI in healthcare
TOPIC: AI in healthcare
QUESTIONS GENERATED: 10

1. What are the primary applications of AI currently used in clinical healthcare settings?
Draft claim: AI is primarily used in clinical settings for medical imaging analysis, predictive patient monitoring, and administrative workflow automation.
Status: unverified_draft

2. How do machine learning algorithms assist radiologists in detecting diseases?
Draft claim: Machine learning algorithms assist radiologists by analyzing medical images to highlight subtle patterns and anomalies indicative of early-stage diseases.
Status: unverified_draft

3. What impact does AI have on drug discovery and development timelines?
Draft claim: AI significantly accelerates drug discovery by rapidly predicting molecular interactions and identifying viable drug candidates.
Status: unverified_draft

4. What are the main ethical concerns regarding algorithmic bias in healthcare AI models?
Draft claim: Algorithmic b

In [20]:
from google.genai import types
from google.genai.errors import ClientError

test_claim = result["questions"][0]["claim"]

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=(
            "Research this claim using web search. "
            "Summarize the evidence and identify useful sources. "
            "Do not treat a search result snippet as conclusive proof.\n\n"
            f"CLAIM: {test_claim}"
        ),
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())]
        )
    )

    print(response.text)

    # Display source URLs returned in the grounding metadata
    for candidate in (response.candidates or []):
        metadata = getattr(candidate, "grounding_metadata", None)
        if metadata:
            for chunk in (getattr(metadata, "grounding_chunks", None) or []):
                web_source = getattr(chunk, "web", None)
                if web_source:
                    print(
                        "SOURCE:",
                        getattr(web_source, "title", "Untitled"),
                        getattr(web_source, "uri", "")
                    )
except ClientError as e:
    if e.code == 429:
        print("API Error: You have exceeded your current Gemini API quota (Rate Limit 429).")
        print("Please wait a minute or two before running this cell again.")
    else:
        print(f"API Client Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

API Error: You have exceeded your current Gemini API quota (Rate Limit 429).
Please wait a minute or two before running this cell again.


In [22]:
import json
import time
from google.genai import types

def research_and_verify(claim):
    prompt = f"""
You are CiteForge's evidence-review assistant.

Use Google Search to find credible evidence relevant to the claim.

CLAIM:
{claim}

Evaluate the evidence you find.

Rules:
- Prefer primary sources, official organizations, peer-reviewed research,
  and authoritative institutions.
- Do not invent sources, quotations, statistics, or URLs.
- A source being relevant does not automatically mean it supports the claim.
- If the evidence is mixed, incomplete, or indirect, use
  "insufficient_evidence".
- Use "supported" only when the retrieved evidence supports the main claim.
- Use "contradicted" only when reliable retrieved evidence conflicts
  with the main claim.
- You are evaluating the retrieved evidence, not proving universal truth.

Return ONLY valid JSON with these keys:
{{
  "claim": "the original claim",
  "verification_status": "supported, contradicted, or insufficient_evidence",
  "explanation": "brief explanation of what the evidence says",
  "evidence": [
    {{
      "source_title": "source title",
      "source_url": "source URL",
      "relevant_finding": "brief summary of relevant evidence"
    }}
  ],
  "limitations": "what remains uncertain"
}}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            response_mime_type="application/json"
        )
    )

    # Parse Gemini's structured text response
    assessment = json.loads(response.text)

    # Collect the actual URLs supplied in search grounding metadata.
    grounded_sources = []
    for candidate in (response.candidates or []):
        metadata = getattr(candidate, "grounding_metadata", None)
        if metadata:
            for chunk in (
                getattr(metadata, "grounding_chunks", None) or []
            ):
                web_source = getattr(chunk, "web", None)
                if web_source:
                    grounded_sources.append({
                        "title": getattr(web_source, "title", "") or "",
                        "url": getattr(web_source, "uri", "") or ""
                    })

    # Keep the original claim and only use URLs actually returned by search.
    assessment["claim"] = claim
    assessment["grounded_sources"] = grounded_sources

    allowed_statuses = {
        "supported",
        "contradicted",
        "insufficient_evidence"
    }

    if assessment.get("verification_status") not in allowed_statuses:
        assessment["verification_status"] = "insufficient_evidence"

    return assessment


verified_results = []

for item in result["questions"]:
    print(f"\nChecking claim {item['id']} of {len(result['questions'])}...")
    print(item["claim"])

    try:
        assessment = research_and_verify(item["claim"])
        assessment["question_id"] = item["id"]
        assessment["question"] = item["question"]
        verified_results.append(assessment)

        print("Verdict:", assessment["verification_status"])
        print("Explanation:", assessment.get("explanation", ""))
        print("Grounded sources found:", len(assessment["grounded_sources"]))

    except Exception as e:
        print("Could not check this claim:", str(e))
        verified_results.append({
            "question_id": item["id"],
            "question": item["question"],
            "claim": item["claim"],
            "verification_status": "insufficient_evidence",
            "explanation": "The research or parsing step failed.",
            "evidence": [],
            "grounded_sources": [],
            "limitations": str(e)
        })

    # Small pause between requests
    time.sleep(1)

print("\nFinished checking claims.")
print("Total processed:", len(verified_results))


Checking claim 1 of 10...
AI is primarily used in clinical settings for medical imaging analysis, predictive patient monitoring, and administrative workflow automation.
Could not check this claim: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

Checking claim 2 of 10...
Machine learning algorithms assist radiologists by analyzing medical images to highlight subtle patterns and anomalies indicative of early-stage diseases.
Could not check this claim: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exc

In [23]:
import pandas as pd

summary = pd.DataFrame([
    {
        "ID": item["question_id"],
        "Question": item["question"],
        "Claim": item["claim"],
        "Verdict": item["verification_status"],
        "Explanation": item.get("explanation", ""),
        "Sources found": len(item.get("grounded_sources", []))
    }
    for item in verified_results
])

display(summary)

,ID,Question,Claim,Verdict,Explanation,Sources found
0,1,What are the primary applications of AI curren...,AI is primarily used in clinical settings for ...,insufficient_evidence,The research or parsing step failed.,0
1,2,How do machine learning algorithms assist radi...,Machine learning algorithms assist radiologist...,insufficient_evidence,The research or parsing step failed.,0
2,3,What impact does AI have on drug discovery and...,AI significantly accelerates drug discovery by...,insufficient_evidence,The research or parsing step failed.,0
3,4,What are the main ethical concerns regarding a...,Algorithmic bias in healthcare AI risks creati...,insufficient_evidence,The research or parsing step failed.,0
4,5,How can healthcare providers integrate AI tool...,Healthcare providers can integrate AI tools in...,insufficient_evidence,The research or parsing step failed.,0
5,6,How does AI improve personalized medicine for ...,AI advances personalized medicine by synthesiz...,insufficient_evidence,The research or parsing step failed.,0
6,7,What data privacy risks are associated with tr...,Training AI models on health records poses pri...,insufficient_evidence,The research or parsing step failed.,0
7,8,What role do clinicians play in validating AI-...,Clinicians serve as the final decision-makers ...,insufficient_evidence,The research or parsing step failed.,0
8,9,Why is the explainability of AI models a major...,Lack of AI explainability creates a barrier to...,insufficient_evidence,The research or parsing step failed.,0
9,10,How are regulatory agencies currently evaluati...,Regulatory agencies evaluate AI-driven medical...,insufficient_evidence,The research or parsing step failed.,0


In [24]:
from google.colab import files

filename = "citeforge_verified_results.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(verified_results, f, indent=2, ensure_ascii=False)

files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
assert "result" in globals(), "Run your question-generation cell first."
assert "verified_results" in globals(), "Run Milestone 3 first."

print("Topic:", result["topic"])
print("Draft claims:", len(result["questions"]))
print("Reviewed claims:", len(verified_results))

Topic: AI in healthcare
Draft claims: 10
Reviewed claims: 10


In [26]:
import json
import html
from datetime import date
from pathlib import Path

def esc(value):
    """Escape text so it can safely appear in HTML."""
    return html.escape(str(value or ""), quote=True)

topic = result["topic"]
today = date.today().isoformat()

# Map each reviewed claim to its original question.
questions_by_id = {
    item["id"]: item for item in result["questions"]
}

page_sections = []
faq_items = []

for index, item in enumerate(verified_results, start=1):
    question_id = item.get("question_id", index)
    original = questions_by_id.get(question_id, {})

    question = item.get("question") or original.get("question", "Research question")
    claim = item.get("claim") or original.get("claim", "")
    status = item.get("verification_status", "insufficient_evidence")
    explanation = item.get("explanation", "No explanation was returned.")
    limitations = item.get("limitations", "")
    anchor = f"claim-{index}"

    # Only display URLs collected from actual Google Search grounding metadata.
    sources = item.get("grounded_sources", [])
    unique_sources = []
    seen_urls = set()

    for source in sources:
        url = source.get("url", "")
        if url.startswith(("https://", "http://")) and url not in seen_urls:
            seen_urls.add(url)
            unique_sources.append(source)

    source_html = ""
    if unique_sources:
        links = []
        for source in unique_sources:
            links.append(
                '<li><a href="' + esc(source["url"]) +
                '" rel="noopener noreferrer" target="_blank">' +
                esc(source.get("title") or source["url"]) +
                '</a></li>'
            )
        source_html = (
            "<h4>Search sources</h4><ul>" + "".join(links) + "</ul>"
        )
    else:
        source_html = (
            "<p><strong>No grounded source URLs were captured for this claim.</strong> "
            "Treat it as needing further research.</p>"
        )

    page_sections.append(f"""
    <article class="claim" id="{anchor}">
      <p class="eyebrow">Claim {index} · {esc(status.replace("_", " ").title())}</p>
      <h3>{esc(question)}</h3>
      <p class="claim-text">{esc(claim)}</p>
      <h4>Evidence assessment</h4>
      <p>{esc(explanation)}</p>
      {source_html}
      <p class="limitation"><strong>Limitations:</strong> {esc(limitations or "This is an automated preliminary assessment; review the sources before relying on the claim.")}</p>
    </article>
    """)

    # FAQ schema answers mirror the visible claim text, not the verdict.
    faq_items.append({
        "@type": "Question",
        "name": question,
        "acceptedAnswer": {
            "@type": "Answer",
            "text": claim
        }
    })

article_schema = {
    "@context": "https://schema.org",
    "@type": "Article",
    "headline": f"{topic}: evidence and research questions",
    "description": (
        f"Research questions and preliminary evidence assessments about {topic}. "
        "Automated assessments require source review."
    ),
    "dateModified": today,
    "mainEntity": {
        "@type": "Thing",
        "name": topic
    }
}

faq_schema = {
    "@context": "https://schema.org",
    "@type": "FAQPage",
    "mainEntity": faq_items
}

schema_json = json.dumps(
    [article_schema, faq_schema],
    ensure_ascii=False
).replace("</", "<\\/")

page_html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{esc(topic)} | CiteForge Research</title>
<meta name="description" content="{esc('CiteForge research page about ' + topic + ', with preliminary evidence assessments and source links.')}">
<script type="application/ld+json">{schema_json}</script>
<style>
:root {{
  color-scheme: light;
  --bg: #f5f7fb;
  --surface: #ffffff;
  --text: #172033;
  --muted: #5c667a;
  --line: #dce2ec;
  --accent: #2947d8;
}}
* {{ box-sizing: border-box; }}
body {{
  margin: 0; background: var(--bg); color: var(--text);
  font: 17px/1.7 system-ui, -apple-system, Segoe UI, sans-serif;
}}
header, main, footer {{ max-width: 900px; margin: auto; padding: 24px; }}
header {{ padding-top: 38px; }}
.brand {{ color: var(--accent); font-weight: 800; letter-spacing: .04em; }}
h1, h2, h3 {{ line-height: 1.25; }}
h1 {{ font-size: clamp(2rem, 5vw, 3.3rem); margin-bottom: 12px; }}
h2 {{ margin-top: 38px; }}
.lede, .muted {{ color: var(--muted); }}
.notice {{
  background: #fff8dc; border: 1px solid #ead78a;
  padding: 16px; border-radius: 12px;
}}
.claim {{
  background: var(--surface); border: 1px solid var(--line);
  border-radius: 16px; padding: 22px; margin: 20px 0;
  overflow-wrap: anywhere;
}}
.eyebrow {{
  color: var(--accent); font-size: .83rem; font-weight: 750;
  text-transform: uppercase; letter-spacing: .04em;
}}
.claim-text {{ font-size: 1.08rem; font-weight: 600; }}
.limitation {{ color: var(--muted); font-size: .92rem; }}
a {{ color: var(--accent); }}
li {{ margin: 6px 0; }}
footer {{ color: var(--muted); border-top: 1px solid var(--line); }}
</style>
</head>
<body>
<header>
  <div class="brand">CITEFORGE · RESEARCH PAGE</div>
  <h1>{esc(topic)}: what the evidence says</h1>
  <p class="lede">Questions, concise claims, source links, and preliminary evidence assessments.</p>
  <p class="muted">Last generated: {today}</p>
  <div class="notice">
    <strong>Important:</strong> These are automated preliminary assessments, not
    professional advice or definitive fact-checks. Read the linked sources.
    A source being found does not necessarily mean it supports the claim.
  </div>
</header>
<main>
  <h2>Research claims</h2>
  {''.join(page_sections)}
</main>
<footer>
  CiteForge prototype · Generated from research results ·
  No guarantee of AI citation or search visibility.
</footer>
</body>
</html>
"""

output_path = Path("citeforge_page.html")
output_path.write_text(page_html, encoding="utf-8")

print("Created:", output_path.resolve())
print("Claims included:", len(page_sections))
print("File size:", output_path.stat().st_size, "bytes")

Created: /content/citeforge_page.html
Claims included: 10
File size: 19263 bytes


In [27]:
from IPython.display import display, HTML
from google.colab import files

# Show a preview inside the notebook
display(HTML(page_html[:page_html.lower().find("</head>") + 7] +
             "<body><h1>" + esc(topic) +
             "</h1><p>Preview only. Download the HTML file to view the full page.</p></body></html>"))

# Download the complete webpage
files.download("citeforge_page.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
import re
import json
from html.parser import HTMLParser

class PageChecker(HTMLParser):
    def __init__(self):
        super().__init__()
        self.ids = []
        self.scripts = 0

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if "id" in attrs:
            self.ids.append(attrs["id"])
        if tag == "script" and attrs.get("type") == "application/ld+json":
            self.scripts += 1

checker = PageChecker()
checker.feed(page_html)

duplicate_ids = sorted({
    item for item in checker.ids if checker.ids.count(item) > 1
})

match = re.search(
    r'<script type="application/ld\+json">(.*?)</script>',
    page_html,
    re.DOTALL
)

print("HTML parsed: YES")
print("Duplicate IDs:", duplicate_ids or "None")
print("JSON-LD blocks:", checker.scripts)

if match:
    structured_data = json.loads(match.group(1))
    print("JSON-LD valid: YES")
    print("Schema types:", [
        item.get("@type") for item in structured_data
    ])
else:
    print("JSON-LD block not found.")

HTML parsed: YES
Duplicate IDs: None
JSON-LD blocks: 1
JSON-LD valid: YES
Schema types: ['Article', 'FAQPage']
